# **Advanced Particle Mesh Simulation on a Single GPU**

<a href="https://colab.research.google.com/github/DifferentiableUniverseInitiative/JaxPM/blob/main/notebooks/02-Advanced_usage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
# Install JaxPM (and diffrax for the ODE solver). Runs only when the packages
# are missing (e.g. on Colab); an existing local install is left untouched.
try:
    import jaxpm, diffrax  # noqa: F401
except ImportError:
    %pip install -q "jaxpm @ git+https://github.com/DifferentiableUniverseInitiative/JaxPM.git"
    %pip install -q diffrax

In [ ]:
import jax
import jax.numpy as jnp
import jax_cosmo as jc
from functools import partial

from diffrax import (ODETerm, SaveAt, SemiImplicitEuler, ConstantStepSize,
                     diffeqsolve)

from jaxpm.pm import linear_field, lpt
from jaxpm.ode import symplectic_ode
from jaxpm.painting import paint
from jaxpm.distributed import uniform_particles
from jaxpm.plotting import plot_fields_single_projection

### Particle-Mesh dynamics with a symplectic solver

We evolve the particles with JaxPM's `symplectic_ode` (the FastPM drift/kick
operators) integrated by diffrax's `SemiImplicitEuler` (symplectic Euler). The
state is stored as **displacements** `dx` from a uniform grid
(`initial_particles='uniform'`) -- the memory-efficient path, since the uniform
integer grid is only added internally at paint time.

In [ ]:
mesh_shape = [128, 128, 128]
box_size = [128., 128., 128.]
snapshots = jnp.array([0.5, 1.0])


@jax.jit
def run_simulation(omega_c, sigma8):
    cosmo = jc.Planck15(Omega_c=omega_c, sigma8=sigma8)

    k = jnp.logspace(-4, 1, 128)
    pk = jc.power.linear_matter_power(cosmo, k)
    pk_fn = lambda x: jnp.interp(x.reshape([-1]), k, pk).reshape(x.shape)
    initial_conditions = linear_field(mesh_shape, box_size, pk_fn,
                                      seed=jax.random.PRNGKey(0))

    # First-order LPT displacements (particles=None -> displacement field).
    dx, p, _ = lpt(cosmo, initial_conditions, a=0.1, order=1)

    drift, kick = symplectic_ode(mesh_shape, cosmo, initial_particles='uniform')
    sol = diffeqsolve((ODETerm(kick), ODETerm(drift)), SemiImplicitEuler(),
                      t0=0.1, t1=1.0, dt0=0.05, y0=(p, dx), args=cosmo,
                      saveat=SaveAt(ts=snapshots),
                      stepsize_controller=ConstantStepSize())

    # sol.ys = (momenta, displacements) at each saved scale factor.
    return initial_conditions, dx, sol.ys[1]


initial_conditions, lpt_displacements, ode_solutions = run_simulation(0.25, 0.8)
ode_solutions.block_until_ready()

In [ ]:
fields = {"Initial Conditions": initial_conditions,
          "LPT Field": jnp.log10(paint(lpt_displacements,
                                       initial_particles='uniform',
                                       order='cic') + 1)}
for a, dx_a in zip(snapshots, ode_solutions):
    fields[f"a = {float(a):.1f}"] = jnp.log10(
        paint(dx_a, initial_particles='uniform', order='cic') + 1)
plot_fields_single_projection(fields)

### First and Second Order Lagrangian Perturbation Theory (LPT) Displacements

This section introduces **first-order and second-order LPT** simulations, controlled by the `order` argument. First-order LPT captures linear displacements, while second-order LPT includes nonlinear corrections, allowing more accurate modeling of structure formation.


In [ ]:
mesh_shape = [128, 128, 128]
box_size = [128., 128., 128.]


@partial(jax.jit, static_argnums=(2,))
def lpt_simulation(omega_c, sigma8, order=1):
    cosmo = jc.Planck15(Omega_c=omega_c, sigma8=sigma8)
    k = jnp.logspace(-4, 1, 128)
    pk = jc.power.linear_matter_power(cosmo, k)
    pk_fn = lambda x: jnp.interp(x.reshape([-1]), k, pk).reshape(x.shape)
    initial_conditions = linear_field(mesh_shape, box_size, pk_fn,
                                      seed=jax.random.PRNGKey(0))
    dx, _, _ = lpt(cosmo, initial_conditions, a=0.8, order=order)
    return initial_conditions, dx


_, lpt_dx_1 = lpt_simulation(0.25, 0.8, order=1)
_, lpt_dx_2 = lpt_simulation(0.25, 0.8, order=2)

In [ ]:
lpt_fields = {
    "First Order": jnp.log10(
        paint(lpt_dx_1, initial_particles='uniform', order='cic') + 1),
    "Second Order": jnp.log10(
        paint(lpt_dx_2, initial_particles='uniform', order='cic') + 1),
}
plot_fields_single_projection(lpt_fields)

### Absolute particle positions

The same dynamics can be run on **absolute positions** instead of displacements.
We seed the particles on a uniform grid, add the 2LPT displacement, and integrate
the absolute positions with the same symplectic solver (`initial_particles=None`).
Absolute painting is simpler to reason about but uses more memory than the
displacement path above.

In [ ]:
mesh_shape = [128, 128, 128]
box_size = [128., 128., 128.]
snapshots = jnp.array([0.1, 0.5, 1.0])


@jax.jit
def run_simulation(omega_c, sigma8):
    cosmo = jc.Planck15(Omega_c=omega_c, sigma8=sigma8)
    k = jnp.logspace(-4, 1, 128)
    pk = jc.power.linear_matter_power(cosmo, k)
    pk_fn = lambda x: jnp.interp(x.reshape([-1]), k, pk).reshape(x.shape)
    initial_conditions = linear_field(mesh_shape, box_size, pk_fn,
                                      seed=jax.random.PRNGKey(0))

    particles = uniform_particles(mesh_shape)
    dx, p, _ = lpt(cosmo, initial_conditions, particles=particles, a=0.1, order=2)

    drift, kick = symplectic_ode(mesh_shape, cosmo, initial_particles=None)
    sol = diffeqsolve((ODETerm(kick), ODETerm(drift)), SemiImplicitEuler(),
                      t0=0.1, t1=1.0, dt0=0.05, y0=(p, particles + dx),
                      args=cosmo, saveat=SaveAt(ts=snapshots),
                      stepsize_controller=ConstantStepSize())

    return initial_conditions, particles + dx, sol.ys[1]


initial_conditions, lpt_particles, ode_particles = run_simulation(0.25, 0.8)
ode_particles.block_until_ready()

In [ ]:
fields = {"Initial Conditions": initial_conditions,
          "LPT Field": jnp.log10(paint(lpt_particles, order='cic') + 1)}
for a, pos_a in zip(snapshots[1:], ode_particles[1:]):
    fields[f"a = {float(a):.1f}"] = jnp.log10(paint(pos_a, order='cic') + 1)
plot_fields_single_projection(fields)

### Weighted projection (displacements)

`paint` accepts a per-particle `weight` array (matching the particle grid). Here
we boost the density in the central region of the box by 10x to highlight it,
compared with the unweighted field.

In [ ]:
center = slice(mesh_shape[0] // 4, 3 * mesh_shape[0] // 4)
center3d = (slice(None), center, center)
weights = jnp.ones(mesh_shape).at[center3d].multiply(10)

weighted = jnp.log10(paint(ode_solutions[0], initial_particles='uniform',
                           order='cic', weight=weights) + 1)
unweighted = jnp.log10(paint(ode_solutions[0], initial_particles='uniform',
                             order='cic') + 1)
plot_fields_single_projection({"Weighted": weighted, "Unweighted": unweighted},
                              project_axis=0)

### Weighted projection (absolute positions)

The same `weight` argument works in absolute mode. Because the weights are tied
to absolute positions (not displacements), the weighted region tracks where the
particles have actually moved to.

In [ ]:
center = slice(mesh_shape[0] // 4, 3 * mesh_shape[0] // 4)
center3d = (slice(None), center, center)
weights = jnp.ones(mesh_shape).at[center3d].multiply(5)

weighted = jnp.log10(paint(ode_particles[0], grid_mesh=jnp.zeros(mesh_shape),
                           order='cic', weight=weights) + 1)
unweighted = jnp.log10(paint(ode_particles[0], grid_mesh=jnp.zeros(mesh_shape),
                             order='cic') + 1)
plot_fields_single_projection({"Weighted": weighted, "Unweighted": unweighted},
                              project_axis=0)